installation des librairies non disponibles dans google colab

In [ ]:
!pip install praat-parselmouth
!pip install pyworld
!pip install IPython
!pip install praat-textgrids
!pip install soundfile


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 79.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.0/261.0 kB 4.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pyworld: filename=pyworld-0.3.5-cp312-cp312-linux_x86_64.whl size=943069 sha256=cc6e7706984d129a2a7409aeac98968f9884e8021469b0b546e13eef16e0a5a9
  Stored in directory: /root/.cache/pip/wheels/be/ac/58/c6a1791ec6d17f3a99b6ccdec92b472f560cb5c564b83dd77e
Successfully built pyworld
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 20.8 MB/s eta 0:00:00


importation des librairies

In [ ]:
import numpy as np
import pyworld as pw
import textgrids
import soundfile as sf
import parselmouth
import matplotlib.pyplot as plt
import librosa
import librosa.display
from parselmouth.praat import call
import IPython.display as ipd
from IPython.display import Audio
from collections import defaultdict


# 1-script principal synthese WORLD

In [ ]:

"""
Modèle phonème-to-speech — LG007, session2_mic_L
"""
import numpy as np
import pyworld as pw
import textgrids
import soundfile as sf
from collections import defaultdict

SR    = 44100
PAS   = 0.005
NOM   = "PTSVOX_LG007_F_session2_mic_L_1"

PHONEMES_VALIDES = {
    'aa', 'ai', 'an', 'au', 'bb', 'ch', 'dd', 'ei', 'eu',
    'ff', 'gg', 'ii', 'in', 'jj', 'kk', 'll', 'mm', 'nn',
    'oe', 'on', 'oo', 'ou', 'pp', 'rr', 'ss', 'tt', 'uu',
    'uy', 'vv', 'ww', 'yy', 'zz'
}

# ---- Analyse WORLD ----
signal, _ = sf.read(NOM + ".wav")
signal     = signal.astype(np.float64)
f0, t      = pw.harvest(signal, SR)
f0         = pw.stonemask(signal, f0, t, SR)
sp         = pw.cheaptrick(signal, f0, t, SR)
ap         = pw.d4c(signal, f0, t, SR)

# ---- Construction du dictionnaire ----
#Un defaultdict(list) est un dictionnaire qui crée automatiquement une liste vide
# quand on accède à une clé qui n'existe pas encore.
#Sans ça il faudrait écrire if p not in dico_sp: dico_sp[p] = [] avant chaque append. C'est juste du confort. On crée 4 dictionnaires en une ligne, un par type de données.

dico_sp, dico_ap, dico_f0, dico_dur = defaultdict(list), defaultdict(list), defaultdict(list), defaultdict(list)

tier = textgrids.TextGrid(NOM + ".corr.textgrid")["phonemes_c"]
#tg   = textgrids.TextGrid(NOM + ".corr.textgrid")
#tier = tg["phonemes_c"]


for a in tier:
    p = a.text.strip()
    if p not in PHONEMES_VALIDES:
        continue
    fd = int(a.xmin / PAS)
    ff = int(a.xmax / PAS)
    #sp[fd:ff] extrait les frames WORLD du phonème courant — c'est une matrice de
    # shape (n_frames, n_bins). La boucle ajoute chaque frame individuellement dans
    # la liste dico_sp[p]. À la fin on aura toutes les frames de toutes les occurrences
    # du phonème p dans la même liste.
    #Pour f0 en revanche, on ne stocke pas chaque frame mais une seule valeur par
    # occurrence : la moyenne de F0 sur le segment. C'est un choix de simplification.

    for frame in sp[fd:ff]: dico_sp[p].append(frame)
    for frame in ap[fd:ff]: dico_ap[p].append(frame)
    dico_f0[p].append(np.mean(f0[fd:ff]))
    dico_dur[p].append(a.xmax - a.xmin)


#une boucle qui crée un dictionnaire. Pour chaque phonème p, on calcule
#la moyenne de toutes les frames accumulées. Le axis=0 signifie qu'on moyenne frame par frame —
#donc si on a 50 occurrences de aa, on fait la moyenne des 50 frames correspondantes pour obtenir
#une seule frame SP moyenne représentative du phonème.
dico = {}
for p in dico_sp:
    dico[p] = {
        "sp":  np.mean(dico_sp[p],  axis=0),
        "ap":  np.mean(dico_ap[p],  axis=0),
        "f0":  np.mean(dico_f0[p]),
        "dur": np.mean(dico_dur[p]),
    }

print(f"Dictionnaire : {sorted(dico.keys())}")



Dictionnaire : ['aa', 'ai', 'an', 'au', 'bb', 'ch', 'dd', 'ei', 'eu', 'ff', 'gg', 'ii', 'in', 'jj', 'kk', 'll', 'mm', 'nn', 'oe', 'on', 'oo', 'ou', 'pp', 'rr', 'ss', 'tt', 'uu', 'uy', 'vv', 'ww', 'yy', 'zz']


In [ ]:

# ---- Synthèse ----
def synthetiser(sequence):
    f0_tot, sp_tot, ap_tot = [], [], []
    for p in sequence:
        #On calcule combien de frames correspond à la durée moyenne du phonème.
        #Par exemple si dur = 0.08s et PAS = 0.005s, alors n = 16 frames.
        #Le max(1, ...) garantit qu'on a au moins 1 frame.
        #n = max(1, int(dico[p]["dur"] / PAS))
        # duree issue de praat et non du dico
        n = max(1, int(dur / PAS))
        #[dico[p]["f0"]] * n crée une liste de n fois la même valeur de F0
        # .extend() ajoute ces valeurs à la liste globale.
        #extend ajoute les éléments d’une liste un par un, alors que append ajoute
        #la liste entière comme un seul bloc.
        #f0_tot.extend([dico[p]["f0"]] * n)
        if f0_val > 0:
          f0_tot.extend([f0_val] * n)
        else:
          f0_tot.extend([0] * n)


        sp_tot.extend([dico[p]["sp"]] * n)
        ap_tot.extend([dico[p]["ap"]] * n)
    # WORLD attend des arrays, pas des listes Python
    return pw.synthesize(np.array(f0_tot), np.array(sp_tot), np.array(ap_tot), SR)


# ---- Exemples ----
y = synthetiser(["bb", "on", "zz", "ou", "rr"])
sf.write("synthese_bonjour.wav", y, SR)


# 2-phonétisation

In [ ]:
texte = 'bonjour'

#phonetisation_praat = call
synthese_praat = call("Create SpeechSynthesizer", "French (France)", "Steph")


grille_synthetisee,son_synthetise = call(synthese_praat,"To Sound", texte, "yes")

# Récupérer le signal audio et le taux d'échantillonnage
signal = son_synthetise.values[0]
sr     = son_synthetise.sampling_frequency
pitch = son_synthetise.to_pitch(time_step=0.01, pitch_floor = 75, pitch_ceiling = 500)


nb_phonemes = call(grille_synthetisee, "Get number of intervals", 4)

#phonemes = [parselmouth.praat.call(grille_synthetisee, "Get label of interval", 4, i + 1) for i in range(nb_phonemes)]
#print(phonemes)

resultats = []

for i in range(1, nb_phonemes + 1):
  label = call(grille_synthetisee, "Get label of interval", 4, i)
  if label =="":
    continue
  xmin = call(grille_synthetisee, "Get start time of interval", 4, i)
  xmax = call(grille_synthetisee, "Get end time of interval", 4, i)
  dur= xmax-xmin
  milieu = (xmin+xmax)/2
  f0_val = pitch.get_value_at_time(milieu)
  if f0_val is None:
    f0_val = 0

  resultats.append({
      "phoneme": label,
      "duree": dur,
      "f0": f0_val
  })

print(resultats)

[{'phoneme': 'b', 'duree': 0.015, 'f0': 175.0531117333906}, {'phoneme': 'ɔ̃', 'duree': 0.081, 'f0': 177.5249471745338}, {'phoneme': 'ʒ', 'duree': 0.076, 'f0': 182.95590807722658}, {'phoneme': 'u', 'duree': 0.194, 'f0': 172.78622794531228}, {'phoneme': 'ʁ', 'duree': 0.04999999999999999, 'f0': 159.5884109539487}]


##############################################
# Jonction entre les 2

In [ ]:

mapping = {
    "b": "bb",
    "ɔ̃": "on",
    "ʒ": "jj",
    "u": "ou",
    "ʁ": "rr"
}
sequence = []
durees = []
f0s = []

for r in resultats:
  ph = r["phoneme"]
  p = mapping[ph]
  sequence.append(p)
  durees.append(r["duree"])
  f0s.append(r["f0"])


print(resultats)

[{'phoneme': 'b', 'duree': 0.015, 'f0': 175.0531117333906}, {'phoneme': 'ɔ̃', 'duree': 0.081, 'f0': 177.5249471745338}, {'phoneme': 'ʒ', 'duree': 0.076, 'f0': 182.95590807722658}, {'phoneme': 'u', 'duree': 0.194, 'f0': 172.78622794531228}, {'phoneme': 'ʁ', 'duree': 0.04999999999999999, 'f0': 159.5884109539487}]


# avec les transitions entre phonèmes

In [ ]:
"""
Fonction
"""
def synthetiser(sequence):
    f0_tot, sp_tot, ap_tot = [], [], []

    for p in sequence:
        n = max(1, int(dico[p]["dur"] / PAS))

        # rééchantillonnage de la trajectoire
        idx = np.linspace(0, len(dico[p]["sp"]) - 1, n).astype(int)

        sp = dico[p]["sp"][idx]
        ap = dico[p]["ap"][idx]
        f0 = np.full(n, dico[p]["f0"])

        f0_tot.extend(f0)
        sp_tot.extend(sp)
        ap_tot.extend(ap)

    return pw.synthesize(
        np.array(f0_tot),
        np.array(sp_tot),
        np.array(ap_tot),
        SR
    )



In [ ]:
"""
script principal
"""
import numpy as np
import pyworld as pw
import textgrids
import soundfile as sf
from collections import defaultdict

SR    = 44100
PAS   = 0.005
NOM   = "PTSVOX_LG007_F_session1_mic_S_1"

PHONEMES_VALIDES = {
    'aa', 'ai', 'an', 'au', 'bb', 'ch', 'dd', 'ei', 'eu',
    'ff', 'gg', 'ii', 'in', 'jj', 'kk', 'll', 'mm', 'nn',
    'oe', 'on', 'oo', 'ou', 'pp', 'rr', 'ss', 'tt', 'uu',
    'uy', 'vv', 'ww', 'yy', 'zz'
}

# ---- Analyse WORLD ----
signal, _ = sf.read(NOM + ".wav")
signal     = signal.astype(np.float64)
f0, t      = pw.harvest(signal, SR)
f0         = pw.stonemask(signal, f0, t, SR)
sp         = pw.cheaptrick(signal, f0, t, SR)
ap         = pw.d4c(signal, f0, t, SR)

# ---- Construction du dictionnaire ----
#Un defaultdict(list) est un dictionnaire qui crée automatiquement une liste vide
# quand on accède à une clé qui n'existe pas encore.
#Sans ça il faudrait écrire if p not in dico_sp: dico_sp[p] = [] avant chaque append. C'est juste du confort. On crée 4 dictionnaires en une ligne, un par type de données.

dico_sp, dico_ap, dico_f0, dico_dur = defaultdict(list), defaultdict(list), defaultdict(list), defaultdict(list)

tier = textgrids.TextGrid(NOM + ".corr.textgrid")["phonemes_c"]
#tg   = textgrids.TextGrid(NOM + ".corr.textgrid")
#tier = tg["phonemes_c"]


for a in tier:
    p = a.text.strip()
    if p not in PHONEMES_VALIDES:
        continue
    fd = int(a.xmin / PAS)
    ff = int(a.xmax / PAS)
    #sp[fd:ff] extrait les frames WORLD du phonème courant — c'est une matrice de
    # shape (n_frames, n_bins). La boucle ajoute chaque frame individuellement dans
    # la liste dico_sp[p]. À la fin on aura toutes les frames de toutes les occurrences
    # du phonème p dans la même liste.
    #Pour f0 en revanche, on ne stocke pas chaque frame mais une seule valeur par
    # occurrence : la moyenne de F0 sur le segment. C'est un choix de simplification.

    for frame in sp[fd:ff]: dico_sp[p].append(frame)
    for frame in ap[fd:ff]: dico_ap[p].append(frame)
    dico_f0[p].append(np.mean(f0[fd:ff]))
    dico_dur[p].append(a.xmax - a.xmin)


#une boucle qui crée un dictionnaire. Pour chaque phonème p, on calcule
#la moyenne de toutes les frames accumulées. Le axis=0 signifie qu'on moyenne frame par frame —
#donc si on a 50 occurrences de aa, on fait la moyenne des 50 frames correspondantes pour obtenir
#une seule frame SP moyenne représentative du phonème.


dico = {}

N_FRAMES = 10  # taille fixe des trajectoires (très simple)

for p in dico_sp:
    traj_sp = []
    traj_ap = []

    # pour chaque occurrence du phonème
    # on concatène toutes les frames (SP/AP) de toutes les occurrences du phonème
    # dans deux grandes trajectoires, puis initialise des listes qui serviront à
    # calculer une version moyennée découpée en N_FRAMES segments
    for i in range(len(dico_sp[p])):
        traj_sp.append(dico_sp[p][i])
        traj_ap.append(dico_ap[p][i])

    traj_sp = np.array(traj_sp)
    traj_ap = np.array(traj_ap)

    # on découpe en N_FRAMES morceaux et on moyenne
    sp_mean = []
    ap_mean = []

    # On crée N_FRAMES+1 indices régulièrement espacés entre 0 et la longueur de traj_sp,
    #convertis en entiers, afin de découper la trajectoire en N_FRAMES segments.
    indices = np.linspace(0, len(traj_sp), N_FRAMES + 1, dtype=int)

    for i in range(N_FRAMES):
        start, end = indices[i], indices[i+1]
        if end > start:
            sp_mean.append(np.mean(traj_sp[start:end], axis=0))
            ap_mean.append(np.mean(traj_ap[start:end], axis=0))
        else:
            sp_mean.append(traj_sp[start])
            ap_mean.append(traj_ap[start])

    dico[p] = {
        "sp": np.array(sp_mean),
        "ap": np.array(ap_mean),
        "f0": np.mean(dico_f0[p]),
        "dur": np.mean(dico_dur[p]),
    }


In [ ]:

# ---- Exemples ----
y = synthetiser(["ss", "oo", "ll", "ei", "jj"])
sf.write("synthese_soleil.wav", y, SR)


# Autres modifications de qualité vocale
a- nasalité


In [ ]:
import numpy as np
import pyworld as pw
import soundfile as sf
import IPython.display as ipd
import matplotlib.pyplot as plt


# ---------- chargement ----------
x, sr = sf.read("PTSVOX_LG008_H_session2_mic_L_1_extraits.wav")
x = x.astype(np.float64)

# ---------- analyse WORLD ----------
f0, t = pw.harvest(x, sr)
sp = pw.cheaptrick(x, f0, t, sr)
ap = pw.d4c(x, f0, t, sr)

# ---------- modification nasale ----------

sp_nasal = sp.copy()
ap_nasal = ap.copy()

# axe fréquentiel
freqs = np.linspace(0, sr / 2, sp.shape[1])

# anti-formant nasal (creux large)
centre = 1300
largeur = 400

masque = 1 - 0.6 * np.exp(-0.5 * ((freqs - centre) / largeur) ** 2)
plt.plot(freqs, masque, label="masque (creux)")

sp_nasal *= masque

# plus de bruit (souffle nasal)
ap_nasal = np.clip(ap_nasal * 1.5, 0, 1)

#for i in range(sp.shape[0]):
#    sp_smooth[i] = np.convolve(sp[i], np.ones(15)/15, mode="same")


# ---------- synthèse ----------
y_nasal = pw.synthesize(f0, sp_nasal, ap_nasal, sr)

# ---------- écoute ----------
ipd.display(ipd.Audio(x, rate=sr))
ipd.display(ipd.Audio(y_nasal, rate=sr))


b- diminution d'une zone (cf. nasalité)

In [ ]:
centre = 1500
largeur = 1000
profondeur = 5.7

notch = 1 - profondeur * np.exp(-0.5 * ((freqs - centre)/largeur)**2)
sp_notch = sp * notch

y = pw.synthesize(f0, sp_notch, ap, sr)

ipd.display(ipd.Audio(y, rate=sr))



c- amplification d'une zone fréquentielle

In [ ]:
centre = 3000
largeur = 500
gain = 5.8

#des filtres spectraux (masque et boost) basés sur une gaussienne centrée sur une fréquence donnée.

boost = 1 + (gain - 1) * np.exp(-0.5 * ((freqs - centre)/largeur)**2)
sp_boost = sp * boost

y = pw.synthesize(f0, sp_boost, ap, sr)

ipd.display(ipd.Audio(y, rate=sr))




# Créer une “voix hybride”

Objectif : combiner source et filtre de deux locuteurs.

In [ ]:
import soundfile as sf

wav1 = "PTSVOX_LG008_H_session2_mic_L_1_extraits.wav"   # source pour F0 et AP
wav2 = "PTSVOX_LG007_F_session2_mic_L_1_extrait2.wav"   # source pour le filtre spectral
out_wav = "synthese_mixte.wav"

# ---------- chargement ----------
x1, sr1 = sf.read(wav1)
x2, sr2 = sf.read(wav2)

#assert sr1 == sr2, "Les deux fichiers doivent avoir le même sampling rate"
#sr = sr1

# mono si besoin
#if x1.ndim > 1:
#    x1 = x1.mean(axis=1)
#if x2.ndim > 1:
#    x2 = x2.mean(axis=1)

x1 = x1.astype(np.float64)
x2 = x2.astype(np.float64)

# ---------- analyse WORLD ----------
f0_1, t1 = pw.harvest(x1, sr)
sp_1 = pw.cheaptrick(x1, f0_1, t1, sr)
ap_1 = pw.d4c(x1, f0_1, t1, sr)

f0_2, t2 = pw.harvest(x2, sr)
sp_2 = pw.cheaptrick(x2, f0_2, t2, sr)
ap_2 = pw.d4c(x2, f0_2, t2, sr)

# ---------- alignement longueur ----------
n_frames = min(len(f0_2), len(sp_1), len(ap_1))

f0_loc2 = f0_2[:n_frames]
sp_loc1 = sp_1[:n_frames]
ap_loc1 = ap_1[:n_frames]

# ---------- synthèse ----------
y = pw.synthesize(f0_loc2, sp_loc1, ap_loc1, sr)

# ---------- écoute ----------
ipd.display(ipd.Audio(y, rate=sr))

#y = pw.synthesize(f0_loc1, sp_loc2, ap_loc1, sr)


# réhausser tout le spectre

In [ ]:
# Cette fois on garde F0, mais on change l’enveloppe spectrale sp.
# Option simple et très parlante :
# On déplace les formants en comprimant l’axe fréquentiel.

sp_mod = sp.copy()
#sp_mod = sp * 2
#sp_mod[:, 200:] *= 0.5


# Compression spectrale (facteur < 1 = voix plus grave/tube plus long)

facteur = 3  # <1 = voix plus "grande", >1 = plus "petite"
# np.arange(0, 10, 2)
for i in range(sp.shape[0]):
    indices = np.arange(sp.shape[1])
    new_indices = indices * facteur
    #new_indices[new_indices >= sp.shape[1]-1] = sp.shape[1]-1
    sp_mod[i] = np.interp(indices, new_indices, sp[i])

y_filter = pw.synthesize(f0, sp_mod, ap, sr)
Audio(y_filter, rate=sr)


## Extension — rendu TTS (projet final)

Cette section a été **ajoutée** pour le rendu : le notebook original du cours n’a pas été réécrit.

- **Synthèse**: concaténation de clips phonème extraits des alignements MFA (`aligned_one/*.TextGrid`) + crossfade + re-synthèse WORLD légère (`projet_tts.py`).
- **Phrases**: 5 phrases de démonstration (hors poème) listées dans `rendu_TTS/manifest.json`.
- **Fichiers générés**: `rendu_TTS/synthese_01.wav` … + `TextGrid` associés.

> Si tu exécutes ce notebook ailleurs (Colab), installe les dépendances comme dans la partie cours (`pyworld`, `soundfile`, etc.).


In [ ]:
from pathlib import Path
import json
import soundfile as sf

rendu = Path('rendu_TTS')
print('manifest:', json.loads((rendu / 'manifest.json').read_text(encoding='utf-8'))[0].keys())

wav = rendu / 'synthese_01.wav'
x, sr = sf.read(wav, always_2d=False)
print(wav, 'sr=', sr, 'samples=', len(x))
